# debug_syntax_deriver_db_2

In [ ]:
from pathlib import Path
import pandas as pd

import param
import panel as pn

from source.evaluate_model import get_syntax_deriver
from source.evaluate_model.ModelEvaluator import get_prompt, get_reply

In [ ]:
pn.extension()

In [ ]:
def reset_syntax_deriver_db(syntax_deriver):
    syntax_deriver.syntax_deriver_db.conn.execute('DELETE FROM rule_errors;')
    syntax_deriver.syntax_deriver_db.conn.execute('DELETE FROM math_statements;')

In [ ]:
def derive_syntax(syntax_deriver, wff, context):
    syntax_deriver.derive_syntax(statement=wff, context=context)

In [ ]:
def get_dataframe(query, conn):
    df = pd.read_sql_query(query, conn)
    return df

In [ ]:
corpus_folder_path = Path('corpus').resolve()
print(f'corpus_folder_path: {corpus_folder_path}')

In [ ]:
# terminal_token = '<|over|>'
# syntax_deriver = get_syntax_deriver(corpus_folder_path=corpus_folder_path)
# prompt = get_prompt()
# # wff = '( E. x A. y ( y e. x <-> E. x ( x e. w /\\ A e. y ) ) <-> E. y A. x ( x e. y <-> ( A F x <-> E. y ( y e. z /\\ E. x ( x e. w /\\ A F y ) ) ) ) )'
# # wff = '( ( ( ( ( ps -> ps ) -> ( -. -. ps -> -. ps ) ) -> -. ps ) -> ps ) -> ph ) -> ps ) -> ( ( ps /\ ps ) -> ( -. ps -> -. ps ) ) )'  # this has a wff followed by more tokens
# # wff = '( ( ( ( ( ps -> ps ) -> ( -. -. ps -> -. ps ) ) -> -. ps ) -> ps ) -> ph )'  # this is a wff
# # wff = '( A e. V -> [_ A / x ]_ { C } = { [_ [_ A / x ]_ C } )'
# wff = r'( ( B e. V /\ x = B ) -> ( x e. A /\ ( A i^i B ) )'
# predicted_statement = f'|- {wff} {terminal_token}'
# wff_statement = get_reply(dictum=predicted_statement, terminal_token=terminal_token)
# context = '\n'.join([prompt, predicted_statement])
# syntax_deriver.derive_syntax(statement=wff_statement, context=context)
# is_ok = syntax_deriver.syntaxDerivation is not None
# print(f'is_ok={is_ok}')

# syntax_deriver

In [ ]:
terminal_token = '<|over|>'
syntax_deriver = get_syntax_deriver(corpus_folder_path=corpus_folder_path)

# derive_syntax

In [ ]:
prompt = get_prompt()
wff = r'( ( B e. V /\ x = B ) -> ( x e. A /\ ( A i^i B ) )'
wff = r"( Fun `' F = dom F -> ( `' A |` ran F ) )"
reset_syntax_deriver_db(syntax_deriver)
derive_syntax(syntax_deriver, wff, context=None)
pn.Column(
    pn.pane.Markdown('# math_statements'),
    pn.pane.DataFrame(get_dataframe("SELECT * FROM math_statements", syntax_deriver.syntax_deriver_db.conn)),
    pn.pane.Markdown('# rule_errors'),
    pn.pane.DataFrame(get_dataframe("SELECT * FROM rule_errors", syntax_deriver.syntax_deriver_db.conn)),
)

# Report

In [ ]:
from source.shared import SyntaxDeriverValidationReporter

def print_context(context):
    parts = context.split('\n')
    if len(parts) >= 2:
        print(f'prompt: {parts[0]}')
        print(f'predicted_statement: {parts[1]}')

max_print_error = 3
max_print_ok = 3
validation_reporter = SyntaxDeriverValidationReporter(syntax_deriver_db=syntax_deriver.syntax_deriver_db, block_size=150)
validation_reporter.print_validation_report(max_print_error=max_print_error, max_print_ok=max_print_ok, print_context=print_context)

# Report 2

In [ ]:
# def colorize(text, color):
#     # return f'<span style="color:{color}">{text}</span>'
#     return text
#
# def phi(syntax_deriver):
#     dynamic_container = pn.Column(margin=(0, 0, 0, 0))
#
#     conn = syntax_deriver.syntax_deriver_db.conn
#     cursor = conn.cursor()
#     sql = 'SELECT id, statement, context, derivation, derivation_correct_count, syntax_deriver_error FROM math_statements ORDER BY id'
#     cursor.execute(sql)
#     math_statement_rows = cursor.fetchall()
#     for math_statement_index in range(len(math_statement_rows)):
#         math_statement_row = math_statement_rows[math_statement_index]
#         statement_id = math_statement_row.id
#         statement = math_statement_row.statement
#         context = math_statement_row.context
#         derivation = math_statement_row.derivation
#         derivation_correct_count = math_statement_row.derivation_correct_count
#         syntax_deriver_error = math_statement_row.syntax_deriver_error
#         prompt = get_prompt()
#         lines = []
#         lines.append(f'prompt: {prompt}')
#         lines.append(f'predicted_statement: {statement}')
#         lines.append(f'error: {syntax_deriver_error}')
#         lines.append(f'derivation_correct_count={derivation_correct_count}')
#         dynamic_container.append(pn.pane.Str('\n'.join(lines)))
#         # print(f'statement: {statement}')
#
#         if derivation is None:
#             lines = []
#             lines.append(f'--- Possible continuations ---')
#             dynamic_container.append(pn.pane.Str('\n'.join(lines)))
#             sql = f'SELECT statement_id, rule_name, rule, mark_index, rule_tokens, current_rule_tokens FROM rule_errors WHERE statement_id = {statement_id} ORDER BY id'
#             cursor.execute(sql)
#             rule_error_rows = cursor.fetchall()
#             for i in range(len(rule_error_rows)):
#                 lines = []
#                 rule_error_row = rule_error_rows[i]
#                 rule_tokens = rule_error_row.rule_tokens
#                 rule_name = rule_error_row.rule_name
#                 rule = rule_error_row.rule
#                 mark_index = rule_error_row.mark_index
#                 # print(f'derivation_correct_count={derivation_correct_count}')
#                 # print(f'mark_index: {mark_index}')
#                 token_index = derivation_correct_count
#                 # print(f'token_index={token_index}')
#                 current_rule_tokens = rule_error_row.current_rule_tokens
#                 # print(f'current_rule_tokens={current_rule_tokens}')
#                 current_token_index = max(0, derivation_correct_count + mark_index)
#                 # print(f'current_token_index={current_token_index}')
#                 statement_split = statement.split()
#                 rule_split = rule.split()
#                 accumulated = " ".join(statement_split[:token_index]) + " "
#                 peeked = " ".join(statement_split[token_index: current_token_index]) + " "
#                 # current_token = statement[current_token_index]
#                 # current_token = f'{statement[current_token_index]} '
#                 current_token = " ".join(statement_split[current_token_index: current_token_index+1]) + " "
#                 rest = " ".join(statement_split[current_token_index+1:])
#                 colorize_text = f'{colorize(text=accumulated, color="#0B6E3F")}{colorize(text=peeked, color="blue")}{colorize(current_token, "red")}{colorize(text=rest, color="black")}'
#                 print(colorize_text)
#                 print(f'accumulated: {accumulated}')
#                 print(f'current_token: {current_token}')
#                 print(f'peeked: {peeked}')
#                 print(f'rest: {rest}')
#                 lines.append(f'Rule {rule_name}: {rule} mark_index={mark_index}')
#                 lines.append(f'Expected: {rule_split[mark_index]}')
#                 subpanel = pn.Column(
#                     # pn.pane.Str(f'===== Example {i + 1} error: ? ====='),
#                     pn.pane.Str('\n'.join(lines)),
#                     pn.pane.Str(colorize_text),
#                     # pn.pane.Markdown(
#                     #     # Metamath color is Color(red: 0.933, green: 1, blue: 0.98, alpha: 1) red: EF green: FF blue: FB '#effffb'
#                     #     colorize_text, styles={'font-family': 'monospace', 'font-size': '12pt', 'background-color': '#effffb', 'padding': '0px 10px'},
#                     # ),
#                 )
#                 dynamic_container.append(subpanel)
#
#     outer_style = {
#         # 'background': 'black',
#         'border': '2px solid blue',
#         'padding': '10px',
#         'margin': "0px",
#     }
#
#     report = pn.Column (
#         dynamic_container,
#         styles=outer_style,
#     )
#
#     return report
#
# phi(syntax_deriver)

# Dec 5, 2025

In [ ]:
def colorize2(text, color):
    return f'<span style="color:{color}">{text}</span>'

In [ ]:
# %%time
# text = r"( Fun `' F = dom F -> ( `' A |` ran F ) )"
# text_split = text.split(' ')
# colorized_text = [colorize2(x, 'red') for x in text_split]
# print(text_split)
# print(colorized_text)
# dynamic_container = pn.Row(margin=(0, 0, 0, 0))
# derivation_correct_count = 4
# mark_index = 2
# for index, item in enumerate(text_split):
#     color = 'black'
#     if index < derivation_correct_count:
#         color = 'blue'
#     elif index < derivation_correct_count + mark_index:
#         color = 'green'
#     elif index == derivation_correct_count + mark_index:
#         color = 'red'
#     dynamic_container.append(pn.pane.Str(colorize2(item, color), styles={'font-family': 'monospace', 'font-size': '12pt', 'background-color': '#effffb', 'padding': '0px 0px'}))
# pn.Column(
#     dynamic_container,
#     styles={'background-color': '#effffb'}
# )

In [ ]:
def tau(text: str, derivation_correct_count: int, mark_index: int):
    dynamic_container = pn.Row(margin=(0, 0, 0, 0))
    text_split = text.split()
    for index, item in enumerate(text_split):
        color = 'gray'
        if index < derivation_correct_count:
            color = 'blue'
        elif index < derivation_correct_count + mark_index:
            color = 'green'
        elif index == derivation_correct_count + mark_index:
            color = 'red'
        dynamic_container.append(pn.pane.Str(colorize2(item, color), styles={'font-family': 'monospace', 'font-size': '12pt', 'background-color': '#effffb', 'padding': '0px 0px'}))
    result = pn.Column(
        dynamic_container,
        # styles={'font-family': 'monospace', 'font-size': '12pt', 'background-color': '#effffb'},
        styles={'background-color': '#effffb'}
    )
    return result

text = r"( Fun `' F = dom F -> ( `' A |` ran F ) )"
derivation_correct_count = 4
mark_index = 2
print(text)
tau(text, derivation_correct_count, mark_index)

In [ ]:
def phi2(syntax_deriver):
    dynamic_container = pn.Column(margin=(0, 0, 0, 0))

    conn = syntax_deriver.syntax_deriver_db.conn
    cursor = conn.cursor()
    sql = 'SELECT id, statement, context, derivation, derivation_correct_count, syntax_deriver_error FROM math_statements ORDER BY id'
    cursor.execute(sql)
    math_statement_rows = cursor.fetchall()
    for math_statement_index in range(len(math_statement_rows)):
        math_statement_row = math_statement_rows[math_statement_index]
        statement_id = math_statement_row.id
        statement = math_statement_row.statement
        context = math_statement_row.context
        derivation = math_statement_row.derivation
        derivation_correct_count = math_statement_row.derivation_correct_count
        syntax_deriver_error = math_statement_row.syntax_deriver_error
        prompt = get_prompt()
        lines = []
        lines.append(f'prompt: {prompt}')
        lines.append(f'predicted_statement: {statement}')
        lines.append(f'error: {syntax_deriver_error}')
        lines.append(f'derivation_correct_count={derivation_correct_count}')
        dynamic_container.append(pn.pane.Str('\n'.join(lines)))
        # print(f'statement: {statement}')

        if derivation is None:
            lines = []
            lines.append(f'--- Possible continuations ---')
            dynamic_container.append(pn.pane.Str('\n'.join(lines)))
            sql = f'SELECT statement_id, rule_name, rule, mark_index, rule_tokens, current_rule_tokens FROM rule_errors WHERE statement_id = {statement_id} ORDER BY id'
            cursor.execute(sql)
            rule_error_rows = cursor.fetchall()
            for i in range(len(rule_error_rows)):
                lines = []
                rule_error_row = rule_error_rows[i]
                rule_tokens = rule_error_row.rule_tokens
                rule_name = rule_error_row.rule_name
                rule = rule_error_row.rule
                mark_index = rule_error_row.mark_index
                # print(f'derivation_correct_count={derivation_correct_count}')
                # print(f'mark_index: {mark_index}')
                token_index = derivation_correct_count
                # print(f'token_index={token_index}')
                current_rule_tokens = rule_error_row.current_rule_tokens
                # print(f'current_rule_tokens={current_rule_tokens}')
                current_token_index = max(0, derivation_correct_count + mark_index)
                # print(f'current_token_index={current_token_index}')
                statement_split = statement.split()
                rule_split = rule.split()
                accumulated = " ".join(statement_split[:token_index]) + " "
                peeked = " ".join(statement_split[token_index: current_token_index]) + " "
                # current_token = statement[current_token_index]
                # current_token = f'{statement[current_token_index]} '
                current_token = " ".join(statement_split[current_token_index: current_token_index+1]) + " "
                rest = " ".join(statement_split[current_token_index+1:])
                colorize_text = f'{colorize2(text=accumulated, color="#0B6E3F")}{colorize2(text=peeked, color="blue")}{colorize2(current_token, "red")}{colorize2(text=rest, color="gray")}'
                # print(colorize_text)
                # print(f'accumulated: {accumulated}')
                # print(f'current_token: {current_token}')
                # print(f'peeked: {peeked}')
                # print(f'rest: {rest}')
                lines.append(f'Rule {rule_name}: {rule} mark_index={mark_index}')
                lines.append(f'Expected: {rule_split[mark_index]} Got: {current_token}')
                subpanel = pn.Column(
                    # pn.pane.Str(f'===== Example {i + 1} error: ? ====='),
                    pn.pane.Str('\n'.join(lines)),
                    pn.pane.Str(colorize_text),
                    # pn.pane.Markdown(
                    #     # Metamath color is Color(red: 0.933, green: 1, blue: 0.98, alpha: 1) red: EF green: FF blue: FB '#effffb'
                    #     colorize_text, styles={'font-family': 'monospace', 'font-size': '12pt', 'background-color': '#effffb', 'padding': '0px 10px'},
                    # ),
                )
                dynamic_container.append(subpanel)

    outer_style = {
        'background': 'white',
        'border': '2px solid blue',
        'padding': '10px',
        'margin': "0px",
    }

    report = pn.Column (
        dynamic_container,
        styles=outer_style,
    )

    return report

phi2(syntax_deriver)

In [ ]:
print("ok")